# Demo Topik 2 — Spark Data Processing
**Pendamping seluruh slide DEMONSTRASI pada Deck 02: slide 9, 19, 25, 27, 33, 34, dan 38**

Jalankan cell secara berurutan. Matikan kernel Spark lain sebelum mulai agar resource cluster tersedia.

## Slide 9 — Membuat SparkSession
Driver berjalan pada kernel Jupyter dan meminta executor kepada Spark standalone cluster.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Spark-Data-Processing-Demos")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Application:", spark.sparkContext.appName)

## Dataset bersama — slide 14
Dataset kecil ini dipakai kembali agar hasil setiap transformasi mudah diperiksa secara manual.

In [ ]:
data = [
    (1, "Jakarta",  "Laptop",   2, 12000000),
    (2, "Bandung",  "Mouse",    5,   150000),
    (3, "Jakarta",  "Keyboard", 3,   500000),
    (4, "Surabaya", "Monitor",  2,  2500000),
    (5, "Bandung",  "Laptop",   1, 12000000),
    (6, "Jakarta",  "Mouse",   10,   150000),
]
columns = ["id", "city", "product", "qty", "price"]
df = spark.createDataFrame(data, columns)
df.show(truncate=False)
df.printSchema()

## Slide 19 — Memilih kolom dengan `select()`
Tekankan bahwa `select()` menghasilkan DataFrame baru; schema `df` awal tidak berubah.

In [ ]:
selected_df = df.select("city", "product", "price")
selected_df.show(truncate=False)
print("Kolom df awal:", df.columns)
print("Kolom selected_df:", selected_df.columns)

## Slide 25 — Kondisi majemuk: AND, OR, NOT
Gunakan `&`, `|`, dan `~`; setiap kondisi harus berada di dalam tanda kurung.

In [ ]:
print("AND — Jakarta dan harga > 1 juta")
df.filter(
    (F.col("city") == "Jakarta") & (F.col("price") > 1_000_000)
).show()

print("OR — Jakarta atau Bandung")
df.filter(
    (F.col("city") == "Jakarta") | (F.col("city") == "Bandung")
).show()

print("NOT — selain Jakarta")
df.filter(~(F.col("city") == "Jakarta")).show()

## Slide 27 — Mengurutkan data dengan `orderBy()`
Bandingkan ascending, descending, dan urutan dengan dua kolom. Sorting global dapat memicu shuffle.

In [ ]:
print("Harga terkecil ke terbesar")
df.orderBy(F.col("price").asc()).show()

print("Harga terbesar ke terkecil")
df.orderBy(F.col("price").desc()).show()

print("Kota ascending, lalu harga descending")
df.orderBy(F.col("city").asc(), F.col("price").desc()).show()

## Slide 33 — Menyaring, menghapus, dan mengisi NULL
Dataset slide 14 tidak memiliki NULL, jadi demo menggunakan DataFrame khusus agar perbedaan `isNull`, `drop`, dan `fill` terlihat.

In [ ]:
df_nulls = spark.createDataFrame([
    (101, "Jakarta", "Laptop", 2, 12000000),
    (102, None, "Monitor", 2, 2500000),
    (103, "Bandung", "Mouse", None, 150000),
], columns)

print("Baris dengan city NULL")
df_nulls.filter(F.col("city").isNull()).show()
print("Baris dengan city tidak NULL")
df_nulls.filter(F.col("city").isNotNull()).show()

df_clean = df_nulls.na.drop(subset=["city"])
df_filled = df_nulls.na.fill({"city": "UNKNOWN", "qty": 0})
print("Setelah drop city NULL")
df_clean.show()
print("Setelah fill city dan qty")
df_filled.show()

## Slide 34 — Membuat kategori dengan `when()`
Kondisi diperiksa dari atas ke bawah; urutan ambang menentukan kategori akhir.

In [ ]:
categorized_df = df.withColumn(
    "kategori_harga",
    F.when(F.col("price") >= 5_000_000, "TINGGI")
     .when(F.col("price") >= 1_000_000, "SEDANG")
     .otherwise("RENDAH"),
)
categorized_df.select("product", "price", "kategori_harga").show()

## Slide 38 — Aggregation dengan `groupBy()`
Hitung jumlah transaksi, total quantity, dan total nilai transaksi per kota.

In [ ]:
df.groupBy("city").count().orderBy("city").show()

df_total = df.withColumn("total", F.col("qty") * F.col("price"))
(
    df_total.groupBy("city")
    .agg(
        F.sum("qty").alias("total_qty"),
        F.sum("total").alias("total_value"),
    )
    .orderBy("city")
    .show()
)

## Cleanup
Tutup SparkSession agar executor tersedia untuk notebook berikutnya.

In [ ]:
spark.stop()
print("SparkSession stopped")